# Simulations using the AllSkyObsTable

In this notebook we look at how we can run simulations using an `AllSkyObsTable`, which does not do any filtering on position. This `ObsTable` type is provided to enable comparison with actual survey cadences or to test "ideal" cadences.

To generate a simulation, we need a few pieces of information:

  * The table of observations (`AllSkyObsTable`)
  * The filter information - We will use the LSST filters
  * The noise model - We will use Gaussian noise with a constant standard deviation.
  * The object model - We use a SALT2 model of SNIa

We create each one below.

## Creating Survey Information

The first three pieces of data (table of observations, filter information, and noise model) are part of the survey information.

### AllSkyObsTable

Unlike the other `ObsTable` types, the `AllSkyObsTable` requires only two columns "time" and "filter". We start with a simple table that has one observation per day in each "g" and "r" filters. By definition of the `AllSkyObsTable`, its footprint covers the entire sky.

In [ ]:
import numpy as np

from lightcurvelynx.obstable.allsky_obstable import AllSkyObsTable

mjd_times = np.array([59580.0, 59580.1, 59581.0, 59581.1, 59582.0, 59582.1, 59590.0, 59590.1])
filters = np.array(["g", "r", "g", "r", "g", "r", "g", "r"])
data = {"time": mjd_times, "filter": filters}

obs_table = AllSkyObsTable(data)
obs_table.plot_footprint(depth=5)

### Filter Information

The second piece of survey information that we need to run the simulation is the filter information. By default `AllSkyObsTable` does not have predefined filters. We will want to use the same filters from the survey we are using for comparison. Let's test with LSST's filter set.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup

pb_group = PassbandGroup.from_preset(preset="LSST")
pb_group.plot()

### Noise Model

The final piece of survey information that we need is a noise model. In this example, we use Gaussian noise with a constant standard deviation (5.0 nJy).

In [ ]:
from lightcurvelynx.noise_models.base_noise_models import ConstantFluxNoiseModel

noise_model = ConstantFluxNoiseModel(5.0)

Given this information we can create a `SurveyInfo` object that wraps everything we want to simulate about the survey.

In [ ]:
from lightcurvelynx.survey_info import SurveyInfo

survey_info = SurveyInfo(
    obstable=obs_table,
    passbands=pb_group,
    noise_model=noise_model,
    survey_name="test_survey",
)

## Creating the Supernova Model

To generate simulated light curves we need to define the properties of the object from which to sample. In this notebook, we use [sncosmo](https://sncosmo.readthedocs.io/en/stable/)'s SALT2 model for Type Ia supernova. 

We start with "sampler" nodes that specify the distribution from which we would like to sample the object's parameters. We use the a uniform sampler on (RA, dec), since we are not filtering on position. We sample redshift uniformly from [0.001, 0.1].

In [ ]:
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import UniformRADEC

ra_dec_sampler = UniformRADEC()
redshift_sampler = NumpyRandomFunc("uniform", low=0.001, high=0.1)

We compute the physical parameters as follows:

In [ ]:
from lightcurvelynx.astro_utils.snia_utils import DistModFromRedshift, X0FromDistMod

# Use the given values for the cosmological parameters (H0=73.0, Omega_m=0.3).
# Then compute the distance modulus from the redshift (taking the redshift sampler as input).
distmod_func = DistModFromRedshift(redshift_sampler, H0=73.0, Omega_m=0.3)

# Sample x1, c, and m_abs from distributions motivated by typical SNIa values.
x1_func = NumpyRandomFunc("normal", loc=0, scale=2.0)
c_func = NumpyRandomFunc("normal", loc=0, scale=0.02)
m_abs_func = NumpyRandomFunc("normal", loc=-19.3, scale=0.1)

# Compute x0 from the other parameters using the standard Tripp formula,
# and the distance modulus from above.
x0_func = X0FromDistMod(
    distmod=distmod_func,  # Use the computed distance modulus from redshift as input.
    x1=x1_func,  # Use the sampled x1 values as input.
    c=c_func,  # Use the sampled c values as input.
    alpha=0.14,  # Use a constant alpha value motivated by typical SNIa fits.
    beta=3.1,  # Use a constant beta value motivated by typical SNIa fits.
    m_abs=m_abs_func,  # Use the sampled m_abs values as input.
)

# t0 for the supernova is sampled uniformly over the time range of the observations.
t0_func = NumpyRandomFunc("uniform", low=np.min(obs_table["time"]), high=np.max(obs_table["time"]))

Note that for more realistic surveys, we would likely want to first sample the host galaxy's properties (using something like [pzflow](https://jfcrenshaw.github.io/pzflow/) to define its parameters) and then sample the SALT2 parameters based on the host's details. In general, LightCurveLynx provides the ability to define a complex directed acyclic graph (DAG) of parameters.

We then define the model using the `SncosmoWrapperModel` class. All of the parameters are set using the samplers defined above. We add linear extrapolation for times defined outside the sncosmo model's bounds.

In [ ]:
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.utils.extrapolate import LinearDecay

source = SncosmoWrapperModel(
    "salt2-h17",  # Model name
    t0=t0_func,
    x0=x0_func,
    x1=x1_func,
    c=c_func,
    ra=ra_dec_sampler.ra,
    dec=ra_dec_sampler.dec,
    redshift=redshift_sampler,
    node_label="source",
    time_extrapolation=LinearDecay(50.0),
)

## Generating the simulations

We can now generate random simulations with all the information defined above. The light curves are written in the [nested-pandas](https://github.com/lincc-frameworks/nested-pandas) format for easy analysis. 

In [ ]:
from lightcurvelynx.simulate import simulate_lightcurves

lightcurves = simulate_lightcurves(
    source,  # The model we are simulating.
    100,  # The number of simulations to run,
    survey_info,  # The survey information
)

# Show the first few entries (dropping the params column for display purposes).
lightcurves.drop(columns=["params"]).head()

Now let's plot the first three random light curves. For reference we will also plot the underlying curve.

In [ ]:
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve
from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.utils.plotting import plot_lightcurves

for idx in range(3):
    # Extract the row for this object.
    lc = lightcurves.iloc[idx]

    # Unpack the nested columns (filters, mjd, flux, and flux error). This is the
    # data from the simulation itself.
    lc_filters = np.asarray(lc["lightcurve"]["filter"], dtype=str)
    lc_mjd = np.asarray(lc["lightcurve"]["mjd"], dtype=float)
    lc_flux = np.asarray(lc["lightcurve"]["flux"], dtype=float)
    lc_fluxerr = np.asarray(lc["lightcurve"]["fluxerr"], dtype=float)

    # Extract the parameters used during the simulation of this object (stored in the "params" column).
    # Use that to compute the noise-free light curve for this object, which we will plot alongside the
    # simulated light curve.
    noise_free_lcs = compute_single_noise_free_lightcurve(
        source,
        GraphState.from_dict(lc["params"]),
        pb_group,
        rest_frame_phase_min=-50.0,  # 50 days before t0
        rest_frame_phase_max=100.0,  # 100 days after t0
        rest_frame_phase_step=0.5,  # 2 samples per day
    )

    plot_lightcurves(
        fluxes=lc_flux,
        times=lc_mjd,
        fluxerrs=lc_fluxerr,
        filters=lc_filters,
        underlying_model=noise_free_lcs,
    )